# Test Trained ViT Encoder Model

This notebook tests models trained with `finetune_with_wandb.py` for multi-task biomedical image processing:
- **Registration**: Image alignment
- **Fusion**: Multi-image fusion
- **Super-Resolution (SR)**: Resolution enhancement
- **Isotropic Restoration (IR)**: Anisotropic to isotropic conversion

**Training Command Used:**
```bash
torchrun --nnodes=1 --nproc_per_node=4 \
    src/finetune_with_wandb.py \
    --model vit \
    --config configs/vit_finetune.yaml \
    --distributed \
    --batch_size 3 \
    --gradient_accumulation_steps 8 \
    --freeze_decoders \
    --amp
```

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import yaml
import sys
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

# Add src to path
sys.path.insert(0, 'src')
sys.path.insert(0, str(Path.cwd()))

# Import config, model and dataset
from orochi.configs.model_configs import ViT3DConfig
from vit_model import ViTULight
from finetune_with_wandb import BiomedicalDataset
import losses

# Set device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

## 2. Load Configuration and Checkpoint

In [ ]:
# Paths - UPDATE THESE
CHECKPOINT_PATH = "/path/to/your/checkpoint.pth"  # UPDATE THIS!

# Create config using ViT3DConfig
config = ViT3DConfig()

# Override with any custom settings if needed
# config.img_size = [64, 256, 256]
# config.batch_size = 1

print("Configuration loaded:")
print(f"  Task: {config.task}")
print(f"  Image size: {config.img_size}")
print(f"  Patch size: {config.patch_size}")
print(f"  Embed dim: {config.embed_dim}")
print(f"  Depths: {config.depths}")
print(f"  Data root: {config.data_root}")

In [ ]:
# Create model using config
print("\nCreating model...")
model = ViTULight(config).to(device)

model.eval()
print(f"Model created with {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")

# Count trainable vs total parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable_params/1e6:.2f}M ({100*trainable_params/total_params:.1f}%)")

In [ ]:
# Load checkpoint
print(f"\nLoading checkpoint from: {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

# Load state dict
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Checkpoint loaded:")
    print(f"  Epoch: {checkpoint.get('epoch', 'N/A')}")
    print(f"  Best loss: {checkpoint.get('best_loss', 'N/A'):.4f}")
elif 'state_dict' in checkpoint:
    model.load_state_dict(checkpoint['state_dict'])
else:
    model.load_state_dict(checkpoint)

print("✓ Checkpoint loaded successfully!")

## 3. Load Test Dataset

In [ ]:
# Load validation dataset
print("\nLoading validation dataset...")

# Create validation dataset
val_dataset = BiomedicalDataset(
    data_root=config.data_root,
    datasets=['hipsc_3d', 'hipsc_2d', 'hipct_2d', 'idr_2d'],
    img_size=config.img_size,
    split='val',
    val_split=0.1
)

# Create DataLoader
val_loader = DataLoader(
    val_dataset,
    batch_size=1,  # Use batch_size=1 for visualization
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print(f"Validation set: {len(val_dataset)} samples")
print(f"Validation batches: {len(val_loader)}")

# Get a sample batch
sample_batch = next(iter(val_loader))
print(f"\nSample batch shape: {sample_batch['image'].shape}")
print(f"Batch keys: {list(sample_batch.keys())}")

## 4. Define Evaluation Metrics

In [ ]:
def compute_psnr(img1, img2, max_val=1.0):
    """Compute Peak Signal-to-Noise Ratio."""
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * torch.log10(max_val / torch.sqrt(mse))

def compute_ssim(img1, img2):
    """Compute Structural Similarity Index (simplified 2D version)."""
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2
    
    mu1 = torch.mean(img1)
    mu2 = torch.mean(img2)
    
    sigma1_sq = torch.var(img1)
    sigma2_sq = torch.var(img2)
    sigma12 = torch.mean((img1 - mu1) * (img2 - mu2))
    
    ssim = ((2 * mu1 * mu2 + C1) * (2 * sigma12 + C2)) / \
           ((mu1**2 + mu2**2 + C1) * (sigma1_sq + sigma2_sq + C2))
    
    return ssim

def compute_ncc(img1, img2):
    """Compute Normalized Cross-Correlation."""
    img1_norm = (img1 - img1.mean()) / (img1.std() + 1e-8)
    img2_norm = (img2 - img2.mean()) / (img2.std() + 1e-8)
    return torch.mean(img1_norm * img2_norm)

print("Metric functions defined:")
print("  - PSNR (Peak Signal-to-Noise Ratio)")
print("  - SSIM (Structural Similarity Index)")
print("  - NCC (Normalized Cross-Correlation)")

## 5. Inference Function

In [ ]:
@torch.no_grad()
def run_inference(model, batch, device):
    """Run inference on a batch and return all task outputs."""
    model.eval()
    
    # Move batch to device
    images = batch['image'].to(device)
    
    # Forward pass - returns (logits, aux_loss)
    logits, aux_loss = model(images)
    
    # logits is a dictionary with task-specific outputs
    # Structure: {
    #   'raw': numpy array,
    #   'reg': {'deformed': ..., 'registered': ..., 'flow': ...},
    #   'fus': {'masked_A': ..., 'masked_B': ..., 'fused': ...},
    #   'SR': {'downsampled': ..., 'super_resolved': ...},
    #   'IR': {'noisy': ..., 'restored': ...}
    # }
    
    results = {
        'input': images.cpu(),
        'logits': logits,  # Full logits dict
        'aux_loss': aux_loss,  # Loss values
    }
    
    # Extract task outputs for easy access
    if 'reg' in logits:
        results['registration'] = torch.from_numpy(logits['reg']['registered'])
    if 'fus' in logits:
        results['fusion'] = torch.from_numpy(logits['fus']['fused'])
    if 'SR' in logits:
        results['super_resolution'] = torch.from_numpy(logits['SR']['super_resolved'])
    if 'IR' in logits:
        results['isotropic_restoration'] = torch.from_numpy(logits['IR']['restored'])
    
    return results

print("Inference function defined")

## 6. Visualization Functions

In [ ]:
def visualize_3d_slice(image, slice_idx=None, channel=0, title="", cmap='gray'):
    """Visualize a 2D slice from a 3D volume."""
    if image.dim() == 5:  # [B, C, D, H, W]
        image = image[0, channel]  # Take first batch, specific channel
    elif image.dim() == 4:  # [B, D, H, W]
        image = image[0]
    elif image.dim() == 3:  # [D, H, W]
        pass
    
    # Select middle slice if not specified
    if slice_idx is None:
        slice_idx = image.shape[0] // 2
    
    plt.figure(figsize=(8, 8))
    plt.imshow(image[slice_idx].cpu().numpy(), cmap=cmap)
    plt.title(f"{title} (Slice {slice_idx})")
    plt.colorbar()
    plt.axis('off')
    plt.tight_layout()
    
def visualize_comparison(input_img, output_img, slice_idx=None, channel=0, task=""):
    """Visualize input vs output comparison."""
    if slice_idx is None:
        slice_idx = input_img.shape[2] // 2 if input_img.dim() == 4 else input_img.shape[0] // 2
    
    # Extract slices
    if input_img.dim() == 5:  # [B, C, D, H, W]
        input_slice = input_img[0, channel, slice_idx].cpu().numpy()
        output_slice = output_img[0, channel, slice_idx].cpu().numpy()
    else:
        input_slice = input_img[slice_idx].cpu().numpy()
        output_slice = output_img[slice_idx].cpu().numpy()
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Input
    im1 = axes[0].imshow(input_slice, cmap='gray')
    axes[0].set_title(f"{task} - Input (Slice {slice_idx})")
    axes[0].axis('off')
    plt.colorbar(im1, ax=axes[0])
    
    # Output
    im2 = axes[1].imshow(output_slice, cmap='gray')
    axes[1].set_title(f"{task} - Output (Slice {slice_idx})")
    axes[1].axis('off')
    plt.colorbar(im2, ax=axes[1])
    
    # Difference
    diff = np.abs(input_slice - output_slice)
    im3 = axes[2].imshow(diff, cmap='hot')
    axes[2].set_title(f"{task} - Absolute Difference")
    axes[2].axis('off')
    plt.colorbar(im3, ax=axes[2])
    
    plt.tight_layout()
    plt.show()

print("Visualization functions defined")

## 7. Run Inference on Sample

In [ ]:
# Get a test sample
print("Running inference on a test sample...")
test_batch = next(iter(val_loader))

# Run inference
results = run_inference(model, test_batch, device)

print("\nInference complete!")
print(f"Input shape: {results['input'].shape}")
if results['outputs'] is not None:
    print(f"Output shape: {results['outputs'].shape}")

## 8. Visualize Results by Task

In [ ]:
# Visualize input
print("\n=== Input Visualization ===")
visualize_3d_slice(results['input'], channel=0, title="Input Image - Channel 0")
plt.show()

if results['input'].shape[1] > 1:
    visualize_3d_slice(results['input'], channel=1, title="Input Image - Channel 1")
    plt.show()

In [ ]:
# Visualize Registration results
if 'registration' in results and results['registration'] is not None:
    print("\n=== Registration Results ===")
    visualize_comparison(
        results['input'], 
        results['registration'],
        task="Registration"
    )

In [ ]:
# Visualize Fusion results
if 'fusion' in results and results['fusion'] is not None:
    print("\n=== Fusion Results ===")
    visualize_comparison(
        results['input'],
        results['fusion'],
        task="Fusion"
    )

In [ ]:
# Visualize Super-Resolution results
if 'super_resolution' in results and results['super_resolution'] is not None:
    print("\n=== Super-Resolution Results ===")
    visualize_comparison(
        results['input'],
        results['super_resolution'],
        task="Super-Resolution"
    )

In [ ]:
# Visualize Isotropic Restoration results
if 'isotropic_restoration' in results and results['isotropic_restoration'] is not None:
    print("\n=== Isotropic Restoration Results ===")
    visualize_comparison(
        results['input'],
        results['isotropic_restoration'],
        task="Isotropic Restoration"
    )

## 9. Quantitative Evaluation on Multiple Samples

In [ ]:
# Evaluate on multiple samples
num_samples = 50  # Number of samples to evaluate
metrics_dict = {
    'registration': {'psnr': [], 'ssim': [], 'ncc': []},
    'fusion': {'psnr': [], 'ssim': [], 'ncc': []},
    'super_resolution': {'psnr': [], 'ssim': [], 'ncc': []},
    'isotropic_restoration': {'psnr': [], 'ssim': [], 'ncc': []}
}

print(f"\nEvaluating on {num_samples} samples...")
with torch.no_grad():
    for i, batch in enumerate(tqdm(val_loader, total=min(num_samples, len(val_loader)))):
        if i >= num_samples:
            break
        
        # Run inference
        results = run_inference(model, batch, device)
        input_img = results['input']
        
        # Compute metrics for each task
        for task_name in ['registration', 'fusion', 'super_resolution', 'isotropic_restoration']:
            if task_name in results and results[task_name] is not None:
                output = results[task_name]
                
                # Compute metrics
                psnr = compute_psnr(input_img, output).item()
                ssim = compute_ssim(input_img, output).item()
                ncc = compute_ncc(input_img, output).item()
                
                metrics_dict[task_name]['psnr'].append(psnr)
                metrics_dict[task_name]['ssim'].append(ssim)
                metrics_dict[task_name]['ncc'].append(ncc)

print("\n✓ Evaluation complete!")

In [ ]:
# Display average metrics
print("\n" + "="*60)
print("QUANTITATIVE RESULTS (Average across samples)")
print("="*60)

for task_name, metrics in metrics_dict.items():
    if len(metrics['psnr']) > 0:
        print(f"\n{task_name.upper().replace('_', ' ')}:")
        print(f"  PSNR: {np.mean(metrics['psnr']):.2f} ± {np.std(metrics['psnr']):.2f} dB")
        print(f"  SSIM: {np.mean(metrics['ssim']):.4f} ± {np.std(metrics['ssim']):.4f}")
        print(f"  NCC:  {np.mean(metrics['ncc']):.4f} ± {np.std(metrics['ncc']):.4f}")

print("\n" + "="*60)

## 10. Visualize Metrics Distribution

In [ ]:
# Plot metrics distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metric_names = ['psnr', 'ssim', 'ncc']
metric_labels = ['PSNR (dB)', 'SSIM', 'NCC']

for idx, (metric, label) in enumerate(zip(metric_names, metric_labels)):
    ax = axes[idx]
    
    for task_name, metrics in metrics_dict.items():
        if len(metrics[metric]) > 0:
            ax.hist(metrics[metric], alpha=0.6, label=task_name.replace('_', ' ').title(), bins=20)
    
    ax.set_xlabel(label)
    ax.set_ylabel('Frequency')
    ax.set_title(f'{label} Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Save Predictions (Optional)

In [ ]:
# Save a few predictions for detailed inspection
import h5py
from datetime import datetime

# Create output directory
output_dir = Path("test_outputs")
output_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"\nSaving predictions to: {output_dir}")

# Save a few samples
num_to_save = 5
for i, batch in enumerate(val_loader):
    if i >= num_to_save:
        break
    
    results = run_inference(model, batch, device)
    
    # Save as HDF5
    output_file = output_dir / f"prediction_{timestamp}_{i:03d}.h5"
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('input', data=results['input'].numpy())
        
        for task_name in ['registration', 'fusion', 'super_resolution', 'isotropic_restoration']:
            if task_name in results and results[task_name] is not None:
                f.create_dataset(task_name, data=results[task_name].numpy())
    
    print(f"  Saved: {output_file.name}")

print(f"\n✓ Saved {num_to_save} predictions to {output_dir}")

## 12. Summary

In [ ]:
print("\n" + "="*70)
print("TESTING SUMMARY")
print("="*70)
print(f"\nCheckpoint: {Path(CHECKPOINT_PATH).name}")
print(f"Test samples: {num_samples}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
print(f"\nTasks evaluated:")
for task_name, metrics in metrics_dict.items():
    if len(metrics['psnr']) > 0:
        print(f"  ✓ {task_name.replace('_', ' ').title()}")
print("\n" + "="*70)
print("\n✓ Testing complete! Check the visualizations and metrics above.")
print("\nNext steps:")
print("  1. Analyze the metrics for each task")
print("  2. Compare with baseline/pretrained models")
print("  3. Inspect saved predictions for qualitative analysis")
print("  4. Fine-tune hyperparameters if needed")